# 09 — BLAST Search

This notebook demonstrates how to use the BLAST search functionality in PBI-Scope to search unknown sequences against the database.

## Prerequisites

- BLAST databases must be built first (run the pipeline with BLAST rules)
- BLAST+ must be installed in the environment

Hit tables are saved as CSV files under `<results>/09_blast_search/` (see Save All Results).

In [ ]:
import os
from pathlib import Path
from pbi import BlastSearcher

# Results directory
results_root = Path(os.getenv('PBI_RESULTS_DIR', '/results'))
results_dir = results_root / '09_blast_search'
results_dir.mkdir(parents=True, exist_ok=True)

# Initialize with default data path (or provide explicit path)
searcher = BlastSearcher()
print("BlastSearcher initialized")
print(f"Results dir  : {results_dir}")

## Check Available Databases

In [ ]:
databases = searcher.list_databases()
for name, info in databases.items():
    status = "READY" if info["exists"] else "NOT BUILT"
    size_gb = info.get('total_size_gb', 0)
    print(f"  [{status}] {name} ({info['type']}) - {size_gb:.2f} GB")

## Database Diagnostics

Check database health and size. Large databases (e.g., >10 GB) may require
longer timeouts for BLAST searches.

In [ ]:
# Get detailed database size information
for name in databases.keys():
    if databases[name]['exists']:
        size = searcher.get_database_size(name)
        size_gb = size / (1024 ** 3)
        print(f"{name}: {size_gb:.2f} GB ({size:,} bytes)")

### Custom Timeout and Threads

For large databases, you may need to increase the timeout or use more threads.
The default timeout is 1800 seconds (30 minutes).

In [ ]:
# Example: search with custom timeout and threads
query = "GTTCTTGTCGAAAAACGTCAACATTTTATAAAAAAGGGTTGCA"

results = searcher.search_sequence(
    query,
    program="blastn",
    db="phages",
    max_hits=5,
    evalue=1e-5,
    timeout=1800,  # 30 minutes (default)
    num_threads=1,  # Use more threads for faster searches
)

print(f"Found {len(results)} hits")
if not results.empty:
    results.head()

## Search a Nucleotide Sequence

Search an unknown DNA sequence against phage genomes.

In [ ]:
# Example: search a short sequence
query = "GTTCTTGTCGAAAAACGTCAACATTTTATAAAAAAGGGTTGCA"

results = searcher.search_sequence(
    query,
    program="blastn",
    db="phages",
    max_hits=5,
    evalue=1e-5,
)

print(f"Found {len(results)} hits")
if not results.empty:
    results.head()

## Search a Protein Sequence

Search an unknown protein sequence against phage proteins.

In [ ]:
# Example protein sequence
protein_query = "MWRRLKEYFSFLKHNPDSKMLNMIANLSKIDIDLDKEEIN"

protein_results = searcher.search_sequence(
    protein_query,
    program="blastp",
    db="proteins",
    max_hits=10,
    evalue=1e-3,
)

print(f"Found {len(protein_results)} hits")
if not protein_results.empty:
    protein_results.head()

## Search a FASTA File

Search multiple sequences from a FASTA file.

In [ ]:
# Create a temporary FASTA file for demonstration
import tempfile
from pathlib import Path

fasta_content = ">seq1\nCTTCCCATGGATCGTTTTGAATTCTTATTTTAGGCTTGTTAA\n>seq2\nGTTTCTAATTATTTTTGTTT\n"
fasta_path = Path(tempfile.mktemp(suffix=".fasta"))
fasta_path.write_text(fasta_content)

# Search all sequences
batch_results = searcher.search_fasta(
    fasta_path,
    program="blastn",
    db="phages",
    max_hits=3,
)

print(f"Found {len(batch_results)} total hits")
if not batch_results.empty:
    batch_results.head(10)

# Clean up
fasta_path.unlink()

## Search Combined Database

Search against the combined database (public + private phages). This is useful
when you want to search against everything at once, including your own private data.

In [ ]:
# Search against the combined database
combined_results = searcher.search_sequence(
    query,
    program="blastn",
    db="combined",
    max_hits=5,
    evalue=1e-5,
)

print(f"Found {len(combined_results)} hits against combined database")
if not combined_results.empty:
    combined_results.head()

## Private Data Ingestion

PBI-Scope can ingest private phage and host data from `private_data/` directory.
This section demonstrates the private data structure using the included `test_private` example.

**⚠️ Warning:** The `test_private` dataset is synthetic test data included for demonstration purposes.
Before running the pipeline on real data, rename the folder, empty it, or delete it — otherwise it WILL be ingested alongside your real data. Alternatively, exclude it in your queries with `WHERE Source_DB != 'test_private'`.

For more information, see the [Private Data Ingestion Guide](../docs/guides/private-data-ingestion.md).

In [ ]:
# Explore the test_private example data structure
import os
from pathlib import Path

private_data_dir = Path(os.getenv('PBI_PRIVATE_DATA_DIR', '/private-data'))
print(f"Private data directory: {private_data_dir}")
print(f"Directory exists: {private_data_dir.exists()}")

if private_data_dir.exists():
    print(f"\nContents:")
    for item in sorted(private_data_dir.iterdir()):
        if item.is_dir():
            print(f"  📁 {item.name}/")
            for subitem in sorted(item.iterdir()):
                if subitem.is_dir():
                    print(f"    📁 {subitem.name}/")
                    for f in sorted(subitem.iterdir()):
                        print(f"      📄 {f.name}")
                else:
                    print(f"    📄 {subitem.name}")

### metadata.csv Format

Each private source must have a `metadata.csv` file with the following required columns:
- `Phage_ID`: Unique phage identifier (must exist in phage.fasta)
- `Host_ID`: Host identifier (use `unknown` if no host genome)
- `Host_name`: Host species name (use `unknown` if no host genome)
- `Source_DB`: Must match the source directory name exactly
- `interaction`: Either `temperate` or `virulent`

Additional columns are stored as entity attributes.

In [ ]:
# Display the test_private metadata.csv
import pandas as pd

test_private_dir = private_data_dir / 'test_private'
if test_private_dir.exists():
    metadata_path = test_private_dir / 'metadata.csv'
    if metadata_path.exists():
        metadata = pd.read_csv(metadata_path)
        print("test_private metadata.csv:")
        print("=" * 60)
        display(metadata)
        print(f"\nColumns: {list(metadata.columns)}")
        print(f"Unique Phage_IDs: {metadata['Phage_ID'].nunique()}")
        print(f"Unique Host_IDs: {metadata['Host_ID'].nunique()}")
    else:
        print(f"metadata.csv not found at {metadata_path}")
else:
    print(f"test_private directory not found at {test_private_dir}")

### Host Genomes

Host genomes are stored in the `hosts/` subdirectory with filenames matching the `Host_ID` column.
Files can be in FASTA format (`.fna`, `.fasta`, or `.fa`) and optionally include index files (`.fai`).

In [ ]:
# List host genomes in test_private
hosts_dir = test_private_dir / 'hosts'
if hosts_dir.exists():
    print("Host genomes in test_private/hosts/:")
    print("=" * 60)
    for f in sorted(hosts_dir.iterdir()):
        size_kb = f.stat().st_size / 1024
        print(f"  {f.name} ({size_kb:.1f} KB)")
else:
    print(f"hosts directory not found at {hosts_dir}")

### Private Data Structure Summary

```
private_data/
  <Source_DB>/
    metadata.csv          # REQUIRED - phage-host relationships
    phage.fasta           # REQUIRED - phage genome sequences
    hosts/                # OPTIONAL - host genome files
      <Host_ID>.fna
      <Host_ID>.fna.fai  # Optional index files
```

For more details, see the [Private Data Ingestion Guide](../docs/guides/private-data-ingestion.md).

## Search Private Database

Search only against private phage sequences (your own data).

In [ ]:
# Search against private database only
private_results = searcher.search_sequence(
    query,
    program="blastn",
    db="private",
    max_hits=5,
    evalue=1e-5,
)

print(f"Found {len(private_results)} hits against private database")
if not private_results.empty:
    private_results.head()

## Private Data Duplicate Detection

The pipeline automatically checks private phage sequences against public data
during database creation. Private phages with >99% identity to public sequences
are flagged as duplicates. This helps identify data that may have been derived
from old public database releases with modified IDs.

In [ ]:
import duckdb
from pbi import get_default_paths

paths = get_default_paths()
db_path = paths.get('duckdb') or paths.get('optimized_duckdb')

if db_path:
    conn = duckdb.connect(str(db_path), read_only=True)
    
    # Check for duplicate annotations
    try:
        dup_count = conn.execute(
            "SELECT COUNT(*) FROM fact_phages WHERE is_duplicate_of_public = TRUE"
        ).fetchone()[0]
        
        total_private = conn.execute(
            "SELECT COUNT(*) FROM fact_phages WHERE source_type = 'private'"
        ).fetchone()[0]
        
        print(f"Private phages: {total_private}")
        print(f"Flagged as duplicates of public data: {dup_count}")
        
        if dup_count > 0:
            print("\nDuplicate details:")
            dups = conn.execute("""
                SELECT Phage_ID, Source_DB, duplicate_public_id, 
                       duplicate_pident, duplicate_qcovs
                FROM fact_phages
                WHERE is_duplicate_of_public = TRUE
                ORDER BY duplicate_pident DESC
            """).fetchdf()
            display(dups)
        else:
            print("No duplicates found between private and public data.")
    except Exception as e:
        print(f"Could not check duplicates: {e}")
        print("The duplicate columns may not exist if the database was built")
        print("before the duplicate detection feature was added.")
    finally:
        conn.close()
else:
    print("Database not found. Run the pipeline first.")

## BLAST Program Selection

| Program | Query Type | Database Type | Use Case |
|---------|-----------|---------------|----------|
| `blastn` | Nucleotide | Nucleotide | Find similar phage genomes |
| `blastp` | Protein | Protein | Find functional protein homologs |
| `blastx` | Nucleotide | Protein | Search translated DNA against proteins |
| `tblastn` | Protein | Nucleotide | Search protein against translated DNA |

## Search Host Genomes

Search against bacterial host genomes to find phage-host relationships.

In [ ]:
# Search against host genomes
host_results = searcher.search_sequence(
    query,
    program="blastn",
    db="hosts",
    max_hits=5,
)

print(f"Found {len(host_results)} hits against host genomes")
if not host_results.empty:
    host_results.head()

## Save Results to CSV

In [ ]:
if not results.empty:
    output_path = results_dir / 'blast_results.csv'
    results.to_csv(output_path, index=False)
    print(f"Results saved to {output_path}")

## BLAST via the API

You can also run BLAST searches through the REST API using the `APIClient`.
This is useful when the API server has BLAST installed but your local
environment does not, or when you want to offload computation to a remote server.

The API enforces a **server-side cap of 100 hits** per request to prevent
resource exhaustion. You can set stricter limits via `max_hits` and `evalue`.

In [ ]:
import os
from pbi.api_client import APIClient

# Connect to the API server (adjust URL as needed)
api_url = os.getenv('PBI_API_URL', 'http://localhost:8000')
client = APIClient(api_url)

# Check BLAST status
status = client.blast_status()
print(f"BLAST installed: {status.get('blast_installed', False)}")
print(f"Databases built: {status.get('databases_built', False)}")
print(f"Available: {status.get('available_databases', [])}")

In [ ]:
# List available BLAST databases
dbs = client.list_blast_databases()
for name, info in dbs.items():
    status_str = "READY" if info["exists"] else "NOT BUILT"
    print(f"  [{status_str}] {name} ({info['type']})")

### API BLAST with Stricter Limits

The API allows you to set stricter `max_hits` and `evalue` limits compared
to the local `BlastSearcher`. This is useful for quick surveys or when
searching large query sets.

In [ ]:
# Strict search: few hits, high significance
api_results = client.blast_search(
    sequence=query,
    program="blastn",
    db="phages",
    max_hits=3,          # stricter than default 10
    evalue=1e-10,        # much stricter than default 1e-5
)

print(f"API returned {len(api_results)} hits (max_hits=3, evalue=1e-10)")
if not api_results.empty:
    api_results.head()

### Search Combined Database via API

Search against both public and private data in one query.

In [ ]:
# Search combined database (public + private)
combined_api_results = client.blast_search(
    sequence=query,
    program="blastn",
    db="combined",
    max_hits=5,
)

print(f"Combined DB: {len(combined_api_results)} hits")
if not combined_api_results.empty:
    combined_api_results.head()

### Search Private Database via API

Search only against your private data.

In [ ]:
# Search private database only
private_api_results = client.blast_search(
    sequence=query,
    program="blastn",
    db="private",
    max_hits=5,
)

print(f"Private DB: {len(private_api_results)} hits")
if not private_api_results.empty:
    private_api_results.head()

### Compare Local vs API Results

The same search through both interfaces should return identical results.

In [ ]:
# Compare local BlastSearcher vs API results
local_results = searcher.search_sequence(
    query, program="blastn", db="phages", max_hits=3, evalue=1e-10
)

print("Local BlastSearcher results:")
print(f"  Hits: {len(local_results)}")
if not local_results.empty:
    print(f"  Top hit: {local_results.iloc[0]['sseqid']} ({local_results.iloc[0]['pident']}%)")

print("\nAPI results:")
print(f"  Hits: {len(api_results)}")
if not api_results.empty:
    print(f"  Top hit: {api_results.iloc[0]['sseqid']} ({api_results.iloc[0]['pident']}%)")

# Close the API client
client.close()

## Advanced: Using Extra BLAST Arguments

In [ ]:
# Example: use word size and dust filtering options
advanced_results = searcher.search_sequence(
    query,
    program="blastn",
    db="phages",
    max_hits=5,
    evalue=1e-10,
    extra_args=["-word_size", "11", "-dust", "no"],
)

print(f"Found {len(advanced_results)} hits with advanced parameters")

## Save All Results

Export all BLAST search results to the outputs directory for later analysis.

In [ ]:
# Save all key results to the results directory
saved = []

result_sets = [
    ('blast_phage_results.csv', results),
    ('blast_protein_results.csv', protein_results),
    ('blast_batch_results.csv', batch_results),
    ('blast_host_results.csv', host_results),
    ('blast_combined_results.csv', combined_results),
    ('blast_private_results.csv', private_results),
    ('blast_advanced_results.csv', advanced_results),
]

# API results (if available)
try:
    result_sets.append(('blast_api_results.csv', api_results))
    result_sets.append(('blast_api_combined_results.csv', combined_api_results))
    result_sets.append(('blast_api_private_results.csv', private_api_results))
except NameError:
    pass

for name, df in result_sets:
    if df is not None and not df.empty:
        path = results_dir / name
        df.to_csv(path, index=False)
        saved.append(f"  {name}: {len(df)} hits")

if saved:
    print(f"Saved {len(saved)} result file(s) to {results_dir}:")
    print('\n'.join(saved))
else:
    print("No results to save.")

---
## Summary

This notebook demonstrated sequence similarity search with `BlastSearcher`:

- **Databases** — status checks for `phages`, `proteins`, `hosts`, `private`, and `combined`
- **Searches** — nucleotide (`blastn`), protein (`blastp`), FASTA-file, host, private, and combined queries
- **API access** — the same searches through `APIClient`, capped at 100 hits per request
- **Exports** — hit tables saved as CSV under `<results>/09_blast_search/`

Remember that BLAST databases are built during the pipeline's first run, which is the slowest step — see [How It Works](../docs/guides/how-it-works.md#pipeline-stages).